# Dataset Credibility Audit and Cleaning

This notebook contains the dataset credibility, leakage, visual inspection, and duplicate-image utilities that were previously at the end of `03_Multimodal_Model_Training_FINAL_Enhanced.ipynb`.
Use this notebook before final model training. It is intentionally audit-first and non-destructive by default.

In [ ]:
from pathlib import Path
import json
import re
import shutil
import hashlib
import base64
import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, HTML

PROJECT_DIR = Path.cwd()
DATA_SPLIT_DIR = PROJECT_DIR / "data_splits"
IMAGE_DIR = PROJECT_DIR / "images"
FINAL_ARTIFACT_DIR = PROJECT_DIR / "final_artifacts"
FINAL_ARTIFACT_DIR.mkdir(exist_ok=True)

LABEL_TO_ID = {"real": 0, "fake": 1}
ID_TO_LABEL = {0: "real", 1: "fake"}

print("Project:", PROJECT_DIR)
print("Split dir exists:", DATA_SPLIT_DIR.exists())
print("Image dir exists:", IMAGE_DIR.exists())
print("Artifact dir:", FINAL_ARTIFACT_DIR)

In [ ]:
DATA_SPLIT_DIR = Path("data_splits")

# For manual inspection, prefer the broad multimodal splits so you can see all available records.
# For final model claims, use the image-deleaked split files produced later in this notebook.
def load_split_for_lookup(split_name):
    """
    Load the most complete available split file.
    Priority:
    1) split_multimodal.csv
    2) split.csv
    3) split_valid_images_only.csv
    4) split_valid_images_only_image_deleaked.csv
    """
    candidates = [
        DATA_SPLIT_DIR / f"{split_name}_multimodal.csv",
        DATA_SPLIT_DIR / f"{split_name}.csv",
        DATA_SPLIT_DIR / f"{split_name}_valid_images_only.csv",
        DATA_SPLIT_DIR / f"{split_name}_valid_images_only_image_deleaked.csv",
    ]

    for path in candidates:
        if path.exists():
            df = pd.read_csv(path)
            df["split"] = split_name
            df["loaded_from"] = str(path)
            return df

    raise FileNotFoundError(f"No split file found for {split_name}. Looked in: {candidates}")


def build_lookup_table():
    all_dfs = []

    for split in ["train", "val", "test"]:
        try:
            df = load_split_for_lookup(split)
            all_dfs.append(df)
        except Exception as e:
            print(f"Skipped {split}: {e}")

    lookup_df = pd.concat(all_dfs, ignore_index=True)

    lookup_df["id"] = lookup_df["id"].astype(str)

    if "title" not in lookup_df.columns:
        lookup_df["title"] = ""

    if "label" not in lookup_df.columns:
        lookup_df["label"] = ""

    if "source" not in lookup_df.columns:
        lookup_df["source"] = "unknown"

    if "image_path_primary" in lookup_df.columns:
        lookup_df["image_path_lookup"] = lookup_df["image_path_primary"]
    elif "image_path" in lookup_df.columns:
        lookup_df["image_path_lookup"] = lookup_df["image_path"]
    else:
        lookup_df["image_path_lookup"] = ""

    lookup_df["title"] = lookup_df["title"].fillna("").astype(str)
    lookup_df["label"] = lookup_df["label"].fillna("").astype(str).str.lower().str.strip()
    lookup_df["source"] = lookup_df["source"].fillna("unknown").astype(str).str.lower().str.strip()
    lookup_df["image_path_lookup"] = lookup_df["image_path_lookup"].fillna("").astype(str)
    lookup_df["image_filename"] = lookup_df["image_path_lookup"].apply(lambda p: Path(p).name if p else "")
    lookup_df["image_stem"] = lookup_df["image_path_lookup"].apply(lambda p: Path(p).stem if p else "")
    lookup_df["valid_image_path"] = lookup_df["image_path_lookup"].apply(lambda p: Path(p).exists() if isinstance(p, str) and p else False)

    return lookup_df


lookup_df = build_lookup_table()

print("Lookup table loaded:")
print("Total records:", len(lookup_df))
display(lookup_df[["split", "id", "source", "label", "title", "image_filename", "valid_image_path", "loaded_from"]].head())

In [ ]:
# Dataset credibility overview: rows, labels, source distribution, and image coverage.
summary = lookup_df.groupby("split").agg(
    n_rows=("id", "size"),
    unique_ids=("id", "nunique"),
    valid_images=("valid_image_path", "sum"),
    unique_sources=("source", "nunique"),
).reset_index()
summary["image_coverage_%"] = (100 * summary["valid_images"] / summary["n_rows"]).round(2)
summary.to_csv(FINAL_ARTIFACT_DIR / "dataset_credibility_split_summary.csv", index=False)
display(summary)

label_source = lookup_df.groupby(["split", "source", "label"]).agg(
    n_rows=("id", "size"),
    valid_images=("valid_image_path", "sum"),
).reset_index()
label_source["image_coverage_%"] = (100 * label_source["valid_images"] / label_source["n_rows"]).round(2)
label_source.to_csv(FINAL_ARTIFACT_DIR / "dataset_credibility_source_label_summary.csv", index=False)
display(label_source.sort_values(["split", "source", "label"]).head(30))

## Manual Image/Text Inspection

In [ ]:
def show_record_by_image_id(image_id, max_results=10, show_image=True):
    """
    Search by:
    1) record id
    2) image filename
    3) image filename without extension
    4) any part of image path
    """

    image_id = str(image_id).strip()

    matches = lookup_df[
        (lookup_df["id"].astype(str) == image_id) |
        (lookup_df["image_filename"].astype(str) == image_id) |
        (lookup_df["image_stem"].astype(str) == image_id) |
        (lookup_df["image_path_lookup"].astype(str).str.contains(image_id, case=False, na=False))
    ].copy()

    if matches.empty:
        print(f"No record found for image_id = {image_id}")
        return None

    cols_to_show = [
        "split",
        "id",
        "source",
        "label",
        "title",
        "image_path_lookup",
        "valid_image_path",
        "loaded_from"
    ]

    cols_to_show = [c for c in cols_to_show if c in matches.columns]

    display(matches[cols_to_show].head(max_results))

    if show_image:
        for _, row in matches.head(max_results).iterrows():
            img_path = row["image_path_lookup"]

            print("=" * 90)
            print("ID:", row["id"])
            print("Split:", row["split"])
            print("Source:", row["source"])
            print("Label:", row["label"])
            print("Title:", row["title"])
            print("Image path:", img_path)

            if row["valid_image_path"]:
                img = Image.open(img_path).convert("RGB")
                plt.figure(figsize=(5, 5))
                plt.imshow(img)
                plt.axis("off")
                plt.title(f'id={row["id"]} | label={row["label"]}')
                plt.show()
            else:
                print("Image file not found on disk.")

    return matches

In [ ]:
# Example lookup. Edit this list with suspicious image ids or filenames you want to inspect.
image_ids = ["gossipcop-843170_0_3ae9eb792d", "gossipcop-842373_1_22757645c6", "gossipcop-844053_1_22757645c6"]

all_found = []

for img_id in image_ids:
    result = show_record_by_image_id(img_id, show_image=False)
    if result is not None:
        all_found.append(result)

if all_found:
    found_df = pd.concat(all_found, ignore_index=True)
    display(found_df[["split", "id", "source", "label", "title", "image_path_lookup", "valid_image_path"]])

In [ ]:
def random_image_text_samples(
    n=9,
    split=None,
    source=None,
    label=None,
    valid_images_only=True,
    seed=42,
    title_chars=90
):
    """
    Randomly display image-text-label samples.

    Parameters:
    n: number of samples
    split: 'train', 'val', 'test', or None
    source: e.g. 'gossipcop', 'politifact', or None
    label: 'real', 'fake', or None
    valid_images_only: True to show only existing images
    seed: random seed
    """

    df = lookup_df.copy()

    if split is not None:
        df = df[df["split"].astype(str).str.lower() == str(split).lower()]

    if source is not None:
        df = df[df["source"].astype(str).str.lower() == str(source).lower()]

    if label is not None:
        df = df[df["label"].astype(str).str.lower() == str(label).lower()]

    if valid_images_only:
        df = df[df["valid_image_path"].astype(bool)]

    if df.empty:
        print("No matching records found with these filters.")
        return pd.DataFrame()

    sample_df = df.sample(n=min(n, len(df)), random_state=seed).reset_index(drop=True)

    display_cols = [
        "split",
        "id",
        "source",
        "label",
        "title",
        "image_path_lookup",
        "valid_image_path"
    ]

    display(sample_df[display_cols])

    for i, row in sample_df.iterrows():
        print("=" * 100)
        print(f"Sample {i+1}")
        print("ID:", row["id"])
        print("Split:", row["split"])
        print("Source:", row["source"])
        print("Label:", row["label"])
        print("Title:", row["title"])
        print("Image:", row["image_path_lookup"])

        if bool(row["valid_image_path"]):
            img = Image.open(row["image_path_lookup"]).convert("RGB")
            plt.figure(figsize=(5, 5))
            plt.imshow(img)
            plt.axis("off")

            short_title = row["title"][:title_chars] + ("..." if len(row["title"]) > title_chars else "")
            plt.title(f'{row["label"]} | {row["id"]}\n{short_title}', fontsize=9)
            plt.show()
        else:
            print("Invalid or missing image path.")

    return sample_df

In [ ]:
# Example visual sample. Change filters as needed.
random_image_text_samples(n=6, source="gossipcop", label="real", seed=554)

## Leakage And Duplicate Audits

In [ ]:
# Split leakage checks: ID overlap, normalized-title overlap, and exact image-content overlap.
def normalize_title(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9 ]+", "", text)
    return text

leakage_rows = []
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    da = lookup_df[lookup_df["split"].eq(a)].copy()
    db = lookup_df[lookup_df["split"].eq(b)].copy()
    ids_a, ids_b = set(da["id"].astype(str)), set(db["id"].astype(str))
    titles_a = set(da["title"].map(normalize_title)) - {""}
    titles_b = set(db["title"].map(normalize_title)) - {""}
    leakage_rows.append({
        "split_pair": f"{a}-{b}",
        "id_overlap": len(ids_a & ids_b),
        "normalized_title_overlap": len(titles_a & titles_b),
        "id_overlap_examples": json.dumps(sorted(list(ids_a & ids_b))[:10]),
        "title_overlap_examples": json.dumps(sorted(list(titles_a & titles_b))[:5]),
    })

split_leakage_audit = pd.DataFrame(leakage_rows)
split_leakage_audit.to_csv(FINAL_ARTIFACT_DIR / "dataset_credibility_id_title_overlap_audit.csv", index=False)
display(split_leakage_audit)

In [ ]:
# Duplicate / identical image checker
# Shows ordered list: representative image + duplicate count

def build_lookup_if_needed():
    """
    Uses lookup_df if it already exists.
    Otherwise builds it from train_final / val_final / test_final.
    """
    if "lookup_df" in globals():
        return lookup_df.copy()

    frames = []

    for split_name, var_name in [
        ("train", "train_final"),
        ("val", "val_final"),
        ("test", "test_final"),
    ]:
        if var_name in globals():
            temp = globals()[var_name].copy()
            temp["split"] = split_name
            frames.append(temp)

    if not frames:
        raise RuntimeError(
            "No lookup_df or train_final/val_final/test_final found. "
            "Run the data loading cells first."
        )

    df = pd.concat(frames, ignore_index=True)

    if "image_path_primary" in df.columns:
        df["image_path_lookup"] = df["image_path_primary"]
    elif "image_path" in df.columns:
        df["image_path_lookup"] = df["image_path"]
    else:
        raise RuntimeError("No image path column found.")

    df["id"] = df["id"].astype(str)
    df["title"] = df["title"].fillna("").astype(str) if "title" in df.columns else ""
    df["label"] = df["label"].fillna("").astype(str) if "label" in df.columns else ""
    df["source"] = df["source"].fillna("").astype(str) if "source" in df.columns else "unknown"

    df["image_path_lookup"] = df["image_path_lookup"].fillna("").astype(str)
    df["valid_image_path"] = df["image_path_lookup"].apply(lambda p: Path(p).exists() if p else False)

    return df


def decoded_pixel_hash(image_path):
    """
    Hash the actual decoded image pixels.
    This ignores filename and metadata.
    If two files have the same pixels, they get the same hash.
    """
    try:
        img = Image.open(image_path).convert("RGB")
        arr = np.asarray(img)
        h = hashlib.sha256()
        h.update(str(img.size).encode("utf-8"))
        h.update(arr.tobytes())
        return h.hexdigest()
    except Exception:
        return None


def image_to_html(path, width=130):
    """
    Convert image to HTML thumbnail.
    """
    try:
        img = Image.open(path).convert("RGB")
        img.thumbnail((width, width))
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG")
        encoded = base64.b64encode(buffer.getvalue()).decode()
        return f'<img src="data:image/jpeg;base64,{encoded}" width="{width}"/>'
    except Exception:
        return "Cannot open image"


df_img = build_lookup_if_needed()

df_img = df_img[df_img["valid_image_path"].astype(bool)].copy()
df_img["image_path_lookup"] = df_img["image_path_lookup"].astype(str)

print("Valid image records:", len(df_img))
print("Unique image paths:", df_img["image_path_lookup"].nunique())

unique_paths = pd.DataFrame({"image_path_lookup": sorted(df_img["image_path_lookup"].unique())})
unique_paths["pixel_hash"] = unique_paths["image_path_lookup"].apply(decoded_pixel_hash)
unique_paths = unique_paths.dropna(subset=["pixel_hash"]).copy()

df_hashed = df_img.merge(unique_paths, on="image_path_lookup", how="left")
df_hashed = df_hashed.dropna(subset=["pixel_hash"]).copy()

duplicate_summary = (
    df_hashed
    .groupby("pixel_hash")
    .agg(
        counter_records=("id", "count"),
        counter_unique_files=("image_path_lookup", "nunique"),
        counter_unique_ids=("id", "nunique"),
        representative_image=("image_path_lookup", "first"),
        example_id=("id", "first"),
        example_title=("title", "first"),
        labels=("label", lambda x: ", ".join(sorted(set(map(str, x))))),
        sources=("source", lambda x: ", ".join(sorted(set(map(str, x))))),
        splits=("split", lambda x: ", ".join(sorted(set(map(str, x))))),
    )
    .reset_index()
)

duplicate_summary = duplicate_summary[duplicate_summary["counter_records"] > 1].copy()
duplicate_summary = duplicate_summary.sort_values(["counter_records", "counter_unique_files"], ascending=False).reset_index(drop=True)
duplicate_summary["group_id"] = np.arange(1, len(duplicate_summary) + 1)

print("Number of duplicate image groups:", len(duplicate_summary))
print("Total duplicated records:", int(duplicate_summary["counter_records"].sum()))

df_hashed.to_csv(FINAL_ARTIFACT_DIR / "dataset_credibility_hashed_image_records.csv", index=False)
duplicate_summary.to_csv(FINAL_ARTIFACT_DIR / "dataset_credibility_duplicate_image_groups.csv", index=False)

display(
    duplicate_summary[
        [
            "group_id",
            "counter_records",
            "counter_unique_files",
            "counter_unique_ids",
            "example_id",
            "labels",
            "sources",
            "splits",
            "example_title",
            "representative_image",
        ]
    ].head(20)
)

In [ ]:
# Exact decoded-pixel duplicate overlap across splits.
cross_split_rows = []
for image_hash, group in df_hashed.groupby("pixel_hash"):
    splits = sorted(group["split"].astype(str).unique().tolist())
    if len(splits) <= 1:
        continue
    cross_split_rows.append({
        "pixel_hash": image_hash,
        "splits": "|".join(splits),
        "n_records": len(group),
        "n_unique_ids": group["id"].nunique(),
        "labels": "|".join(sorted(group["label"].astype(str).unique().tolist())),
        "sources": "|".join(sorted(group["source"].astype(str).unique().tolist())),
        "example_ids": json.dumps(group["id"].astype(str).head(10).tolist()),
        "example_paths": json.dumps(group["image_path_lookup"].astype(str).head(5).tolist()),
    })

image_cross_split_leakage = pd.DataFrame(cross_split_rows)
if len(image_cross_split_leakage):
    image_cross_split_leakage = image_cross_split_leakage.sort_values(["n_records", "pixel_hash"], ascending=[False, True]).reset_index(drop=True)
image_cross_split_leakage.to_csv(FINAL_ARTIFACT_DIR / "dataset_credibility_cross_split_image_duplicate_audit.csv", index=False)

pair_rows = []
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    ha = set(df_hashed.loc[df_hashed["split"].eq(a), "pixel_hash"].dropna())
    hb = set(df_hashed.loc[df_hashed["split"].eq(b), "pixel_hash"].dropna())
    overlap = ha & hb
    pair_rows.append({
        "split_pair": f"{a}-{b}",
        "n_overlapping_pixel_hashes": len(overlap),
        "overlap_%_of_smaller_split": round(100 * len(overlap) / max(1, min(len(ha), len(hb))), 3),
    })

image_overlap_pair_summary = pd.DataFrame(pair_rows)
image_overlap_pair_summary.to_csv(FINAL_ARTIFACT_DIR / "dataset_credibility_cross_split_image_duplicate_pair_summary.csv", index=False)
display(image_overlap_pair_summary)
if len(image_cross_split_leakage):
    display(image_cross_split_leakage.head(20))

In [ ]:
# Display duplicate images as ordered visual list

def show_duplicate_image_list(top_n=30, thumbnail_width=140):
    if duplicate_summary.empty:
        print("No duplicate image groups found.")
        return

    view = duplicate_summary.head(top_n).copy()

    view["image"] = view["representative_image"].apply(
        lambda p: image_to_html(p, width=thumbnail_width)
    )

    html_df = view[
        [
            "group_id",
            "image",
            "counter_records",
            "counter_unique_files",
            "counter_unique_ids",
            "labels",
            "sources",
            "splits",
            "example_id",
            "example_title",
        ]
    ].copy()

    html_df = html_df.rename(columns={
        "group_id": "Group",
        "counter_records": "Count Records",
        "counter_unique_files": "Unique Files",
        "counter_unique_ids": "Unique Article IDs",
        "labels": "Labels",
        "sources": "Sources",
        "splits": "Splits",
        "example_id": "Example ID",
        "example_title": "Example Text / Title",
        "image": "Image",
    })

    display(HTML(html_df.to_html(escape=False, index=False)))

show_duplicate_image_list(top_n=50)

## Cleaning Outputs

In [ ]:
# Create de-leaked valid-image split CSVs by removing validation/test rows whose image pixels appeared earlier.
# Train is kept unchanged; val is filtered against train; test is filtered against train plus cleaned val.
def load_valid_split(split):
    candidates = [
        DATA_SPLIT_DIR / f"{split}_valid_images_only.csv",
        DATA_SPLIT_DIR / f"{split}_multimodal.csv",
        DATA_SPLIT_DIR / f"{split}.csv",
    ]
    for path in candidates:
        if path.exists():
            df = pd.read_csv(path)
            df["split"] = split
            return df
    raise FileNotFoundError(f"No split file found for {split}")


def attach_pixel_hash(df, split_name):
    key = df_hashed[["split", "id", "pixel_hash"]].drop_duplicates().copy()
    out = df.copy()
    out["id"] = out["id"].astype(str)
    out["split"] = split_name
    return out.merge(key, on=["split", "id"], how="left")

train_valid = attach_pixel_hash(load_valid_split("train"), "train")
val_valid = attach_pixel_hash(load_valid_split("val"), "val")
test_valid = attach_pixel_hash(load_valid_split("test"), "test")

train_hashes = set(train_valid["pixel_hash"].dropna())
val_keep = ~val_valid["pixel_hash"].isin(train_hashes)
val_deleaked = val_valid[val_keep].reset_index(drop=True)
removed_val = val_valid[~val_keep].reset_index(drop=True)

prior_hashes_for_test = train_hashes | set(val_deleaked["pixel_hash"].dropna())
test_keep = ~test_valid["pixel_hash"].isin(prior_hashes_for_test)
test_deleaked = test_valid[test_keep].reset_index(drop=True)
removed_test = test_valid[~test_keep].reset_index(drop=True)

train_valid.to_csv(DATA_SPLIT_DIR / "train_valid_images_only_image_deleaked.csv", index=False)
val_deleaked.to_csv(DATA_SPLIT_DIR / "val_valid_images_only_image_deleaked.csv", index=False)
test_deleaked.to_csv(DATA_SPLIT_DIR / "test_valid_images_only_image_deleaked.csv", index=False)
removed_val.to_csv(FINAL_ARTIFACT_DIR / "removed_val_image_pixel_leakage_rows.csv", index=False)
removed_test.to_csv(FINAL_ARTIFACT_DIR / "removed_test_image_pixel_leakage_rows.csv", index=False)

deleak_audit = pd.DataFrame([
    {"split": "train", "rows_before": len(train_valid), "rows_after": len(train_valid), "rows_removed_for_prior_split_image_overlap": 0},
    {"split": "val", "rows_before": len(val_valid), "rows_after": len(val_deleaked), "rows_removed_for_prior_split_image_overlap": len(removed_val)},
    {"split": "test", "rows_before": len(test_valid), "rows_after": len(test_deleaked), "rows_removed_for_prior_split_image_overlap": len(removed_test)},
])
deleak_audit.to_csv(FINAL_ARTIFACT_DIR / "dataset_credibility_image_deleakage_split_filter_audit.csv", index=False)
display(deleak_audit)

In [ ]:
# Duplicate train image report, non-destructive by default

DUP_THRESHOLD = 4
ACTION_MODE = "report_only"  # allowed: "report_only", "copy". Moving is intentionally disabled here.
TRAIN_DIR = PROJECT_DIR / "images" / "train"
TRAIN_DUP_DIR = PROJECT_DIR / "images" / "train_dup_review"
TRAIN_DUP_DIR.mkdir(parents=True, exist_ok=True)

selected_groups = duplicate_summary[duplicate_summary["counter_records"] > DUP_THRESHOLD].copy()
selected_hashes = set(selected_groups["pixel_hash"])

dup_records = df_hashed[df_hashed["pixel_hash"].isin(selected_hashes)].copy()
if "split" in dup_records.columns:
    dup_records = dup_records[dup_records["split"].astype(str).str.lower().eq("train")].copy()

dup_records["image_path_lookup"] = dup_records["image_path_lookup"].astype(str)
dup_records["image_path_obj"] = dup_records["image_path_lookup"].apply(Path)
dup_records = dup_records[dup_records["image_path_obj"].apply(lambda p: p.exists())].copy()

def is_inside_train(path):
    try:
        path.resolve().relative_to(TRAIN_DIR.resolve())
        return True
    except Exception:
        return False

dup_records = dup_records[dup_records["image_path_obj"].apply(is_inside_train)].copy()

group_info = selected_groups[["pixel_hash", "group_id", "counter_records", "counter_unique_files", "counter_unique_ids"]].copy()
dup_records = dup_records.merge(group_info, on="pixel_hash", how="left")

cols_to_save = [
    "group_id", "counter_records", "counter_unique_files", "counter_unique_ids",
    "split", "id", "source", "label", "title", "image_path_lookup", "pixel_hash",
]
cols_to_save = [c for c in cols_to_save if c in dup_records.columns]

ids_report_path = FINAL_ARTIFACT_DIR / "duplicate_train_image_ids_count_above_4.csv"
dup_records[cols_to_save].to_csv(ids_report_path, index=False)

print("Selected duplicate groups:", len(selected_groups))
print("Train duplicate records in selected groups:", len(dup_records))
print("Unique image files:", dup_records["image_path_lookup"].nunique() if len(dup_records) else 0)
print("Saved report:", ids_report_path)

processed_files = []
if ACTION_MODE == "copy":
    unique_files = (
        dup_records[["group_id", "counter_records", "pixel_hash", "image_path_lookup"]]
        .drop_duplicates()
        .sort_values(["counter_records", "group_id"], ascending=[False, True])
    )
    for _, row in unique_files.iterrows():
        src = Path(row["image_path_lookup"])
        group_id = int(row["group_id"])
        count = int(row["counter_records"])
        group_folder = TRAIN_DUP_DIR / f"group_{group_id:04d}_count_{count}"
        group_folder.mkdir(parents=True, exist_ok=True)
        dst = group_folder / src.name
        if dst.exists():
            dst = group_folder / f"{src.stem}_{row['pixel_hash'][:8]}{src.suffix}"
        shutil.copy2(str(src), str(dst))
        processed_files.append({
            "group_id": group_id,
            "counter_records": count,
            "pixel_hash": row["pixel_hash"],
            "original_path": str(src),
            "new_path": str(dst),
            "action": "copied",
        })
elif ACTION_MODE != "report_only":
    raise ValueError("ACTION_MODE must be 'report_only' or 'copy'. Moving files is disabled in this notebook.")

processed_df = pd.DataFrame(processed_files)
processed_report_path = FINAL_ARTIFACT_DIR / "duplicate_train_images_processed_count_above_4.csv"
processed_df.to_csv(processed_report_path, index=False)

print("Action mode:", ACTION_MODE)
print("Files copied:", len(processed_df))
print("Processed files report:", processed_report_path)
display(dup_records[cols_to_save].head(50))
if len(processed_df):
    display(processed_df.head(50))